# CAMIA Step 2 & 3: Train Classifier and Generate ROC Curves

**This notebook runs step 2 and 3 of CAMIA using pre-generated features (.pkl file)**

## Prerequisites:
- Upload `all_features_custom.pkl` from step 1 to Kaggle dataset
- This notebook will train the classifier and generate ROC curves
- Total time: ~10 minutes

## Step 0: Setup Environment

In [ ]:
import os
import shutil
import sys

# Set working directory
os.chdir('/kaggle/working')

# Create expected directory structure
results_dir = 'results_new/camia_custom_dataset/common-pile_comma-v0.1-2t'
os.makedirs(results_dir, exist_ok=True)

print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Results directory: {results_dir}")

## Step 1: Copy .pkl File to Expected Location

**IMPORTANT: Update the path below to match where you uploaded your .pkl file**

In [ ]:
# UPDATE THIS PATH to where you uploaded the pkl file
# Examples:
# /kaggle/input/camia-features/all_features_custom.pkl
# /kaggle/input/your-dataset-name/all_features_custom.pkl

PKL_INPUT_PATH = '/kaggle/input/camia-features/all_features_custom.pkl'  # <- CHANGE THIS
PKL_OUTPUT_PATH = f'{results_dir}/all_features_custom.pkl'

# Copy the file
if os.path.exists(PKL_INPUT_PATH):
    shutil.copy(PKL_INPUT_PATH, PKL_OUTPUT_PATH)
    print(f"✓ Copied pkl file to {PKL_OUTPUT_PATH}")
    print(f"✓ File size: {os.path.getsize(PKL_OUTPUT_PATH) / (1024**2):.2f} MB")
else:
    print(f"❌ ERROR: File not found at {PKL_INPUT_PATH}")
    print(f"\nAvailable datasets:")
    os.system('ls -la /kaggle/input/')

## Step 2: Download CAMIA Codebase

In [ ]:
# Clone or download the CAMIA repo
if not os.path.exists('context_aware_mia'):
    os.system('git clone https://github.com/changhongyan123/context_aware_mia.git')
    print("✓ Cloned CAMIA repository")
else:
    print("✓ CAMIA repository already exists")

# Change to CAMIA directory
os.chdir('/kaggle/working/context_aware_mia')
print(f"✓ Working directory: {os.getcwd()}")

## Step 3: Copy .pkl File to CAMIA Results Directory

In [ ]:
# The CAMIA scripts expect the pkl file here
camia_results_dir = f'/kaggle/working/results_new/camia_custom_dataset/common-pile_comma-v0.1-2t'
os.makedirs(camia_results_dir, exist_ok=True)

# Copy pkl from input to CAMIA results directory
shutil.copy(PKL_OUTPUT_PATH, f'{camia_results_dir}/all_features_custom.pkl')
print(f"✓ Copied pkl to CAMIA directory: {camia_results_dir}")

## Step 4: Install Requirements (if needed)

In [ ]:
# Install scikit-learn and other ML libraries if not present
os.system('pip install -q scikit-learn scipy pandas matplotlib shap')
print("✓ Dependencies installed")

## Step 5: Run Training Script (Step 2)

In [ ]:
print("\n" + "="*50)
print("[2/3] Training Logistic Regression Classifier")
print("="*50 + "\n")

os.system('python run_ours_train_lr.py base_model=common-pile/comma-v0.1-2t experiment_name=comma_custom_books')
print("\n✓ Training complete!")

## Step 6: Generate ROC Curves (Step 3)

In [ ]:
print("\n" + "="*50)
print("[3/3] Generating ROC Curves")
print("="*50 + "\n")

os.system('python run_ours_get_roc.py base_model=common-pile/comma-v0.1-2t experiment_name=comma_custom_books')
print("\n✓ ROC generation complete!")

## Step 7: View Results

In [ ]:
import json
import glob

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50 + "\n")

# Find all output files
results_path = f'/kaggle/working/results_new/camia_custom_dataset/common-pile_comma-v0.1-2t'
if os.path.exists(results_path):
    files = os.listdir(results_path)
    print("Generated files:")
    for f in files:
        fpath = os.path.join(results_path, f)
        fsize = os.path.getsize(fpath) / 1024  # KB
        print(f"  ✓ {f} ({fsize:.1f} KB)")
    
    # Try to load and display metrics if JSON exists
    json_files = glob.glob(f'{results_path}/*.json')
    if json_files:
        print(f"\nMetrics from {json_files[0]}:")
        with open(json_files[0], 'r') as f:
            metrics = json.load(f)
            for key, value in metrics.items():
                if isinstance(value, (int, float)):
                    print(f"  {key}: {value:.4f}")
else:
    print(f"❌ Results directory not found: {results_path}")

## Step 8: Download Results

In [ ]:
# Create a zip of all results for download
import zipfile

results_path = f'/kaggle/working/results_new/camia_custom_dataset/common-pile_comma-v0.1-2t'
zip_path = '/kaggle/working/camia_results.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir(results_path):
        file_path = os.path.join(results_path, file)
        zipf.write(file_path, arcname=file)

print(f"✓ Results zipped: {zip_path}")
print(f"✓ File size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")
print("\n✓ Download 'camia_results.zip' from Kaggle Output")

## Done! 🎉

### What was generated:
- ✓ Trained classifier model
- ✓ MIA prediction scores
- ✓ ROC curves and metrics
- ✓ AUC scores (at different FPR thresholds)

### Next steps:
1. Download `camia_results.zip` from Kaggle output
2. Analyze metrics in the JSON files
3. View ROC curves (PNG files)
4. Extract insights from the MIA scores